In [ ]:
import os
import re
import html
import shutil
import hashlib
from pathlib import Path

import pandas as pd
import chardet
import ipywidgets as widgets
from IPython.display import display, HTML
from ipyfilechooser import FileChooser

In [ ]:
# --- EDF header reading (custom binary parser, reused from inspect_edf; no signal is ever loaded) ---

def detect_encoding(byte_string, min_confidence=0.6):
    result = chardet.detect(byte_string)
    encoding = result["encoding"]
    confidence = result["confidence"]
    if encoding is None or confidence < min_confidence:
        raise UnicodeDecodeError("chardet", byte_string, 0, len(byte_string),
                                 f"\tUnable to reliably detect encoding. Detected: {encoding} with confidence {confidence}")
    return encoding


def read_edf_header_custom(file_path):
    with open(file_path, "rb") as f:
        header = {}
        raw_header = f.read(256)
        encoding = detect_encoding(raw_header)
        f.seek(0)
        header["version"] = f.read(8).decode(encoding).strip()
        header["patient_id"] = f.read(80).decode(encoding).strip()
        header["recording_id"] = f.read(80).decode(encoding).strip()
        header["start_date"] = f.read(8).decode(encoding).strip()
        header["start_time"] = f.read(8).decode(encoding).strip()
    return header


def check_anonymization(patient_id, file_stem):
    """
    Inspect the EDF+ 'Local Patient ID' field for a real patient name.
    EDF+ format: 'code sex birthdate name' (space-separated).
    Compumedics encodes the name as 'LASTNAME_FIRSTNAME'; anonymized = 'X_X'.
    The name subfield (4th token onward) is isolated, placeholder chars
    (X/x/_ and whitespace) are stripped; anything remaining is a real name.
    Filename cross-check: each name token with >=3 chars is searched in the
    file stem (case-insensitive) to detect PII leaking into the file name.
    """
    parts = str(patient_id).split()
    if len(parts) >= 4:
        name_subfield = " ".join(parts[3:])
        id_format = "edf+"
    else:
        name_subfield = str(patient_id).strip()
        id_format = "non-standard"
    cleaned = re.sub(r"[Xx_,;\s]", "", name_subfield)
    header_anonymized = (cleaned == "")
    name_in_filename = False
    if not header_anonymized:
        stem_low = file_stem.lower()
        for token in re.split(r"[_\s]+", name_subfield):
            if len(token) >= 3 and token.lower() in stem_low:
                name_in_filename = True
                break
    warning = ""
    if not header_anonymized:
        if name_in_filename:
            warning = "PII in header AND file name"
        else:
            warning = "header NOT anonymized (file name looks clean)"
    return {
        "patient_id"        : str(patient_id).strip(),
        "name_subfield"     : name_subfield,
        "patient_id_format" : id_format,
        "header_anonymized" : header_anonymized,
        "name_in_filename"  : name_in_filename,
        "anon_warning"      : warning,
    }


# --- Anonymization core (writes a NEW file; only header bytes 8..168 are overwritten) ---

ANON_PATIENT_ID = "X X 30-DEC-1899 X_X"   # Compumedics anonymized format
_MONTHS = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN",
           "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]


def build_anon_recording_id(orig_recording_id, start_date_field):
    """Keep the real 'Startdate <dd-MMM-yyyy>' token (recording night is not direct PII),
    blank the trailing admin/technician/equipment fields to 'X X X'. Fall back to the
    start_date header field ('dd.mm.yy') if no Startdate token is present."""
    m = re.search(r"(Startdate\s+\d{1,2}-[A-Za-z]{3}-\d{4})", orig_recording_id)
    if m:
        return f"{m.group(1)} X X X"
    m2 = re.match(r"(\d{2})\.(\d{2})\.(\d{2})", start_date_field.strip())
    if m2:
        dd, mm, yy = m2.groups()
        mon = _MONTHS[int(mm) - 1] if 1 <= int(mm) <= 12 else "XXX"
        year = 2000 + int(yy) if int(yy) < 85 else 1900 + int(yy)   # EDF year-clipping rule
        return f"Startdate {dd}-{mon}-{year} X X X"
    return "Startdate X X X X"


def _write_field(f, offset, text):
    """Overwrite an 80-byte fixed-width header field in place (ASCII, space-padded)."""
    field = text.encode("ascii", errors="replace")[:80].ljust(80)
    f.seek(offset)
    f.write(field)


def read_identity_fields(path):
    with open(path, "rb") as f:
        h = f.read(184)
    return {
        "patient_id":   h[8:88].decode("latin-1").strip(),
        "recording_id": h[88:168].decode("latin-1").strip(),
        "start_date":   h[168:176].decode("latin-1").strip(),
        "start_time":   h[176:184].decode("latin-1").strip(),
    }


def _sha_after_256(path):
    """SHA-256 of everything from byte 256 onward (per-channel headers + signal records)."""
    hsh = hashlib.sha256()
    with open(path, "rb") as f:
        f.seek(256)
        for chunk in iter(lambda: f.read(1 << 20), b""):
            hsh.update(chunk)
    return hsh.hexdigest()


def anonymize_edf_header(src, dst, new_patient_id=ANON_PATIENT_ID, anon_recording_id=True, verify_signal=True):
    """
    Copy src -> dst, then overwrite ONLY the identity fields in the 256-byte general header:
      patient_id   (bytes 8..88)   -> new_patient_id (default Compumedics 'X X 30-DEC-1899 X_X')
      recording_id (bytes 88..168) -> keep real Startdate, blank trailing fields (if anon_recording_id)
    start_date/start_time (168..184) are left untouched. Signal data and per-channel headers
    (bytes 256+) are never modified. Returns a result dict (before/after + verification).
    """
    shutil.copy2(src, dst)
    before = read_identity_fields(dst)
    new_rec = build_anon_recording_id(before["recording_id"], before["start_date"]) if anon_recording_id else before["recording_id"]
    with open(dst, "r+b") as f:
        _write_field(f, 8, new_patient_id)
        if anon_recording_id:
            _write_field(f, 88, new_rec)
    after = read_identity_fields(dst)
    result = {
        "before": before,
        "after": after,
        "size_ok": (Path(src).stat().st_size == Path(dst).stat().st_size),
        "signal_identical": None,
    }
    if verify_signal:
        result["signal_identical"] = (_sha_after_256(src) == _sha_after_256(dst))
    return result


# --- Companion files + filename suggestion ---

def find_companions(edf_path):
    """Files in the same folder sharing the EDF stem at a separator boundary.
    The boundary check (next char after the stem is '.', '_', '-' or space) prevents
    a stem like '73' from matching '731_clipping.edf'."""
    stem = edf_path.stem
    companions = []
    for p in sorted(edf_path.parent.iterdir()):
        if p == edf_path or not p.is_file():
            continue
        name = p.name
        if name.startswith(stem):
            rest = name[len(stem):]
            if rest == "" or rest[0] in "._- ":
                companions.append(p)
    return companions


def suggest_new_stem(stem, info):
    """If the patient name leaks into the filename, propose a stem with the name
    token(s) (>=3 chars) stripped out; otherwise keep the original stem."""
    if not info["name_in_filename"]:
        return stem
    new = stem
    for token in re.split(r"[_\s]+", info["name_subfield"]):
        if len(token) >= 3:
            new = re.sub(re.escape(token), "", new, flags=re.IGNORECASE)
    new = re.sub(r"[_\s]{2,}", "_", new)
    new = re.sub(r"-{2,}", "-", new)
    new = new.strip("_- ")
    return new or stem

## EDF anonymizer &mdash; Jupyter (debug) version

Same tool as `1bis_anonymize_edf_voila.ipynb`, with code cells visible for debugging. Run all cells, then use
the widgets below. Originals are never modified; anonymized copies go into an `anonymized/` sub-folder.

Header fields rewritten: **patient_id** &rarr; `X X 30-DEC-1899 X_X`; **recording_id** &rarr; real `Startdate` kept,
trailing fields &rarr; `X X X`. `start_date` / `start_time` kept. Signal data is byte-for-byte identical.

In [ ]:
fc_folder = FileChooser()
fc_folder.title = "<b>Select your data folder:</b>"
fc_folder.show_only_dirs = True

chk_process_anonymized = widgets.Checkbox(
    value=False,
    description="Also list files already anonymized",
    style={"description_width": "initial"},
)
chk_anon_recording = widgets.Checkbox(
    value=True,
    description="Anonymize recording_id trailing fields (keep the recording date)",
    style={"description_width": "initial"},
)
chk_skip_existing_output = widgets.Checkbox(
    value=True,
    description="Skip files already present in the anonymized/ folder (untick to recompute everything)",
    style={"description_width": "initial"},
)

scan_info = widgets.HTML(value="")
review_header = widgets.HTML(value="")
review_box = widgets.VBox([])

# Holds one dict per reviewable file: {edf, inc (checkbox), stem (text), info, folder}
review_rows = []


def scan_and_build(*_):
    review_rows.clear()
    review_box.children = ()
    review_header.value = ""
    if not fc_folder.selected:
        scan_info.value = ""
        return
    folder = Path(fc_folder.selected)
    edf_files = [f for f in sorted(folder.rglob("*"))
                 if f.suffix.lower() == ".edf" and not f.name.startswith("._")]
    if not edf_files:
        scan_info.value = '<span style="color:#888;">No EDF files found (recursive scan).</span>'
        return

    rows_widgets = []
    n_total = len(edf_files)
    n_nonanon = 0
    n_already = 0
    n_unreadable = 0

    for edf in edf_files:
        try:
            hdr = read_edf_header_custom(edf)
        except Exception as e:
            n_unreadable += 1
            rows_widgets.append(widgets.HTML(
                f'<span style="color:#c0392b;">&#9888; cannot read header of '
                f'<code>{html.escape(edf.name)}</code>: {html.escape(str(e))}</span>'))
            continue
        info = check_anonymization(hdr["patient_id"], edf.stem)
        if info["header_anonymized"]:
            n_already += 1
            if not chk_process_anonymized.value:
                continue
        else:
            n_nonanon += 1

        inc = widgets.Checkbox(value=not info["header_anonymized"], indent=False,
                               layout=widgets.Layout(width="30px"))
        stem = widgets.Text(value=suggest_new_stem(edf.stem, info),
                            layout=widgets.Layout(width="260px"))
        if info["header_anonymized"]:
            badge = '<span style="color:#2e7d32;">already anonymized</span>'
        elif info["name_in_filename"]:
            badge = '<span style="color:#c0392b;font-weight:bold;">name in header AND file name</span>'
        else:
            badge = '<span style="color:#e67e00;">name in header only (file name looks clean)</span>'
        label = widgets.HTML(
            f'<code>{html.escape(edf.name)}</code> &rarr; new name: &nbsp;'
            f'(detected name: <b>{html.escape(info["name_subfield"])}</b>) &nbsp; {badge}')
        rows_widgets.append(widgets.HBox([inc, stem, label]))
        review_rows.append({"edf": edf, "inc": inc, "stem": stem, "info": info, "folder": folder})

    color = "#2e7d32" if n_nonanon == 0 else "#c0392b"
    extra = f' &middot; <span style="color:#c0392b;">{n_unreadable} unreadable</span>' if n_unreadable else ""
    scan_info.value = (
        f'<b>{n_total}</b> EDF files &mdash; '
        f'<b style="color:{color};">{n_nonanon}</b> not anonymized, '
        f'<b style="color:#2e7d32;">{n_already}</b> already anonymized{extra}.')
    if review_rows:
        review_header.value = (
            '<b>Review:</b> tick the files to anonymize; edit the new file name if needed '
            '(the patient name is stripped automatically when it appears in the file name).')
    else:
        review_header.value = '<i>Nothing to anonymize in this folder.</i>'
    review_box.children = tuple(rows_widgets)


fc_folder.register_callback(scan_and_build)
chk_process_anonymized.observe(scan_and_build, names="value")

display(
    HTML('<h3 style="margin-top:4px;">&#128193; Data folder</h3>'),
    fc_folder,
    scan_info,
    HTML('<h3 style="margin-top:16px;">&#9881; Options</h3>'),
    chk_anon_recording,
    chk_skip_existing_output,
    chk_process_anonymized,
    HTML('<h3 style="margin-top:16px;">&#128221; Files to anonymize</h3>'),
    review_header,
    review_box,
)

In [ ]:
btn_run = widgets.Button(
    description="▶  Run anonymization",
    button_style="primary",
    layout=widgets.Layout(width="240px", height="40px"),
)
out = widgets.Output()

FILES_DESCRIPTION = (
    "# anonymized — Output files\n\n"
    "Created by 1bis_anonymize_edf_voila.ipynb (Dream-Toolkit). Contains header-anonymized copies of EDF\n"
    "files. ORIGINAL files are never modified.\n\n"
    "| File | Description |\n"
    "|------|-------------|\n"
    "| <subtree>/<file>.edf | Header-anonymized EDF (patient_id -> 'X X 30-DEC-1899 X_X'; recording_id "
    "trailing fields -> X X X; recording date kept). Signal data identical to the original. |\n"
    "| <subtree>/<companions> | Hypnogram / .edf.XML / event companions, copied and renamed to match. "
    "Content NOT modified. |\n"
    "| anonymization_log.tsv | One row per processed file: before/after identity fields, rename, "
    "companions, verification. |\n\n"
    "WARNING: After verifying this folder, delete the original non-anonymized files yourself.\n"
    "WARNING: Companion file content is copied unchanged - verify your XML/txt carry no patient name.\n"
)


def run_anonymization(btn):
    btn.disabled = True
    out.clear_output()
    try:
        if not fc_folder.selected:
            with out:
                print("Please select a data folder first.")
            return
        if not review_rows:
            with out:
                print("Nothing to process. Select a folder that contains non-anonymized files.")
            return
        included = [r for r in review_rows if r["inc"].value]
        if not included:
            with out:
                print("No files selected. Tick at least one file to anonymize.")
            return

        folder = Path(fc_folder.selected)
        out_root = folder / "anonymized"
        out_root.mkdir(exist_ok=True)

        log_rows = []
        failed = []
        n_done = 0
        n_skipped = 0

        for r in included:
            edf = r["edf"]
            info = r["info"]
            new_stem = (r["stem"].value or "").strip() or edf.stem
            rel = edf.parent.relative_to(folder)
            dst_dir = out_root / rel
            dst = dst_dir / f"{new_stem}.edf"

            # Skip if output already exists (unless the user unticked the option).
            if dst.exists() and chk_skip_existing_output.value:
                n_skipped += 1
                with out:
                    print(f"Skipped (already exists): {dst.relative_to(folder)}")
                continue

            # --- Fatal step: copy + patch header. On failure, record and continue. ---
            try:
                dst_dir.mkdir(parents=True, exist_ok=True)
                res = anonymize_edf_header(edf, dst, anon_recording_id=chk_anon_recording.value)
            except Exception as e:
                failed.append({"original_path": str(edf), "reason": f"header patch: {e}"})
                with out:
                    print(f"❌ {edf.name}: {e}")
                continue

            verified = check_anonymization(res["after"]["patient_id"], new_stem)["header_anonymized"]
            signal_ok = res["signal_identical"]
            size_ok = res["size_ok"]

            # --- Non-fatal step: copy + rename companions. Warn and continue on error. ---
            companions_copied = []
            for comp in find_companions(edf):
                try:
                    new_comp_name = new_stem + comp.name[len(edf.stem):]
                    shutil.copy2(comp, dst_dir / new_comp_name)
                    companions_copied.append(new_comp_name)
                except Exception as e:
                    with out:
                        print(f"⚠ {edf.name}: could not copy companion {comp.name}: {e}")

            log_rows.append({
                "original_path"        : str(edf),
                "anonymized_path"      : str(dst),
                "renamed"              : (new_stem != edf.stem),
                "patient_id_before"    : res["before"]["patient_id"],
                "patient_id_after"     : res["after"]["patient_id"],
                "recording_id_before"  : res["before"]["recording_id"],
                "recording_id_after"   : res["after"]["recording_id"],
                "name_subfield"        : info["name_subfield"],
                "companions_copied"    : "; ".join(companions_copied),
                "signal_identical"     : signal_ok,
                "size_identical"       : size_ok,
                "verified_anonymized"  : verified,
                "status"               : "ok" if (verified and signal_ok and size_ok) else "CHECK",
            })
            n_done += 1
            with out:
                msg = f"✅ {edf.name} → {dst.relative_to(folder)}"
                if companions_copied:
                    msg += f"  (+{len(companions_copied)} companion(s))"
                if not (verified and signal_ok and size_ok):
                    msg += "  ⚠ VERIFICATION FAILED — inspect this file"
                print(msg)

        # --- Write / merge the log (replace rows for files processed this run, on normalized path) ---
        log_path = out_root / "anonymization_log.tsv"
        df_new = pd.DataFrame(log_rows)
        if not df_new.empty:
            if log_path.exists():
                try:
                    df_old = pd.read_csv(log_path, sep="\t", dtype=str)
                    done_norm = {os.path.normcase(p) for p in df_new["original_path"]}
                    df_old = df_old[~df_old["original_path"].apply(lambda p: os.path.normcase(str(p)) in done_norm)]
                    df_new = pd.concat([df_old, df_new], ignore_index=True)
                except Exception as e:
                    with out:
                        print(f"⚠ could not merge with existing log ({e}); writing this run only.")
            df_new.to_csv(log_path, sep="\t", index=False)

        # --- Write the failed list, if any ---
        if failed:
            pd.DataFrame(failed).to_csv(out_root / "anonymization_failed.tsv", sep="\t", index=False)

        # --- FILES_DESCRIPTION.md (consistent with the other tools) ---
        try:
            with open(out_root / "FILES_DESCRIPTION.md", "w", encoding="utf-8") as fdesc:
                fdesc.write(FILES_DESCRIPTION)
        except Exception as e:
            with out:
                print(f"⚠ could not write FILES_DESCRIPTION.md: {e}")

        with out:
            print("\n=== Done ===")
            print(f"Anonymized : {n_done}")
            print(f"Skipped    : {n_skipped}")
            print(f"Failed     : {len(failed)}")
            print(f"Output     : {out_root}")
            if log_rows:
                print(f"Log        : {log_path}")
            print("\n⚠ Originals were NOT modified. After verifying the anonymized/ folder, "
                  "delete the original non-anonymized files yourself.")
            print("⚠ Companion file CONTENT was copied unchanged — verify your XML/txt carry no patient name.")

    except Exception as e:
        with out:
            print(f"Unexpected error — run interrupted: {e}")
    finally:
        btn.disabled = False


btn_run.on_click(run_anonymization)

display(
    HTML('<h3 style="margin-top:20px;">&#9654; Run</h3>'),
    btn_run,
    out,
)